# Football Predictor — Tutorial

This notebook walks you through the system from scratch. No ML background needed.

**Run cells top to bottom.** Each cell builds on the previous one.

---
## Step 1 — Install & Setup

Run this cell first. It installs the package and all dependencies.

In [ ]:
import subprocess, sys, os

project_root = os.path.abspath('..')

# Verify we're on Python 3.11 (project requires 3.11.x ARM64 via Homebrew)
major, minor = sys.version_info.major, sys.version_info.minor
print(f'Kernel Python: {sys.executable}  ({sys.version})')
assert (major, minor) == (3, 11), (
    f'Wrong Python version: {major}.{minor}\n'
    'Go to Kernel → Change Kernel and select Python 3.11.'
)

# Install the package using this kernel's pip
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', project_root, '-q'], check=True)

# Add scripts folder to path (needed for predict_wc2026 / generate_submission imports)
sys.path.insert(0, os.path.join(project_root, 'scripts'))

import warnings
warnings.filterwarnings('ignore')

print('Setup complete.')

---
## Step 2 — The Data

The model trains on international match results going back to 1872 — every World Cup, qualifier, continental tournament, and friendly ever recorded.

Not all matches are equal. A World Cup final tells us more about team quality than a June friendly. So we weight matches:

| Match type | Weight |
|---|---|
| Friendly | 0.3× |
| Nations League / minor | 0.7× |
| Qualifiers / continental | 1.0× |
| FIFA World Cup | 1.5× |

In [ ]:
from football_predictor.data.sources.international_results import fetch_training_data

# Downloads ~47,000 matches on first run, then reads from local cache
data = fetch_training_data(from_year=2010)

print(f'Matches loaded: {len(data):,}')
print(f'Date range:     {data["date"].min()} → {data["date"].max()}')
print()
data[['date','home_team','away_team','home_goals','away_goals','tournament']].tail(5)

---
## Step 3 — Features (what does the model know?)

Before the model can predict, we turn each match into a row of numbers — ~120 features from 13 training modules, after removing near-duplicate columns (|Pearson r| > 0.95).

WC 2026-specific signals (sofifa, api_form, venue, injury, squad list) are applied as **post-processing adjustments** after the ensemble, not as training features — they would cause covariate shift since they're zero for all historical matches.

Think of each training module as one source of information:

| Module | Plain English |
|---|---|
| **Elo** | Simple rating — win = go up, lose = go down. Match importance scales the K-factor. |
| **Glicko-2** | Like Elo but also tracks *how confident* we are in the rating |
| **Kalman Filter** | Tracks attack and defence separately, and how they change over time. EM-tuned process noise. |
| **Form** | Last 5/10/20 matches — recent results, goals scored/conceded |
| **Strength of schedule** | Win rate adjusted for opponent quality |
| **Head-to-head** | How these two teams have historically matched up |
| **Squad strength** | Long-run attack/defence quality from competitive history |
| **Confederation** | Data-derived: CONMEBOL=65 > UEFA=50 > AFC=25 > CAF=10 > CONCACAF=−20 |
| **Tournament stage** | Group vs knockout? Is it a must-win game? |
| **FIFA Rankings** | Official FIFA world ranking points |
| **Odds** | Bookmaker implied probabilities — strongest single external signal |
| **xG form** | Rolling xG/xGA from qualifier data (UEFA/AFC/CONMEBOL; CAF/CONCACAF unavailable) |
| **Transfermarkt** | Squad market values (June 2026) |

In [ ]:
from football_predictor.data.pipeline import build_feature_matrix
from football_predictor.constants import DEFAULT_FEATURE_MODULES

# Filter to competitive matches only (qualifiers, World Cups, continental tournaments)
comp = data[data['tournament'].str.lower().str.contains(
    'qualif|world cup|copa|euro|nations|africa|asian|gold cup', na=False
)].reset_index(drop=True)

print(f'Building features for {len(comp):,} competitive matches...')
X, y = build_feature_matrix(comp, DEFAULT_FEATURE_MODULES, context=data)

print(f'Done: {X.shape[0]:,} matches × {X.shape[1]} features')
print(f'\nFirst 10 feature names:')
print(list(X.columns[:10]))

---
## Step 4 — The Model Stack

We layer five stages on top of each other:

```
~120 training features (after correlation pruning)
     │
     ▼
[XGBoost]                    ← learns which features matter
     │
     ▼
[Temperature Scaling]        ← fixes overconfidence (single scalar T)
     │
[BayesPoisson MAP]           ← estimates expected goals λ per team
     │  Dixon-Coles ρ correction for low scores
     │  half-life derived from Kalman EM-tuned process noise
     ▼
[Context-Adaptive Ensemble]  ← per-match α = sigmoid(w · [odds_avail, kalman_unc, log1p_h2h])
     │
     ▼
[WC 2026 Post-Processing]    ← venue λ adjustment, sofifa quality nudge, player absence
     │
     ▼
P(home win), P(draw), P(away win)
```

**Why five stages?** XGBoost uses all features but doesn't model goals. BayesPoisson models goals structurally. The ensemble weight adapts to match context — trusting XGBoost more when odds are available, BayesPoisson more when Kalman uncertainty is high.

The cell below trains all stages — takes ~60 seconds.

In [ ]:
from predict_wc2026 import train_model

xgb, temp_cal, bp, ensemble, all_data, train_df = train_model(quiet=False)

---
## Step 5 — Predicting a Single Match

Let's predict France vs Norway — one of the tightest Group I matchups.

In [ ]:
from football_predictor.data.wc2026 import normalise

home, away = 'France', 'Norway'

# Expected goals from BayesPoisson
lam_h, lam_a = bp.get_lambdas(normalise(home), normalise(away), neutral=True)
print(f'Expected goals:  {home} = {lam_h:.2f}   {away} = {lam_a:.2f}')

# Win/draw/loss probabilities
bp_p = bp.predict_proba(normalise(home), normalise(away), neutral=True)
print(f'\nOutcome probabilities:')
print(f'  {home} win: {bp_p["home_win"]:.1%}')
print(f'  Draw:       {bp_p["draw"]:.1%}')
print(f'  {away} win: {bp_p["away_win"]:.1%}')

---
## Step 6 — Score Prediction (Optimised for Points)

We don't just pick the most likely score. We pick the score that **maximises expected competition points**:

- 5 pts — exact score
- 3 pts — right winner/draw + right goal difference
- 2 pts — right winner/draw only

For every possible score we compute:
```
E[pts] = 2×P(right outcome) + 1×P(right outcome AND right GD) + 2×P(exact score)
```
Then pick the highest.

In [ ]:
import numpy as np
from generate_submission import _simulate_match, _match_pts

p_home = bp_p['home_win']
p_draw  = bp_p['draw']
p_away  = bp_p['away_win']

rng = np.random.default_rng(42)
sim_h, sim_a = _simulate_match(lam_h, lam_a, p_home, p_draw, p_away, 50_000, rng)

print(f'Top predicted scores for {home} vs {away}:')
print(f'{"Score":<8} {"E[pts]":>7}')
print('-' * 17)

results = []
for h_p in range(5):
    for a_p in range(5):
        ev = _match_pts(h_p, a_p, sim_h, sim_a).mean()
        results.append((h_p, a_p, ev))
results.sort(key=lambda x: -x[2])

for h_p, a_p, ev in results[:8]:
    best = ' ← best' if (h_p, a_p) == results[0][:2] else ''
    print(f'{h_p}–{a_p:<5}   {ev:>5.3f}{best}')

---
## Step 7 — Group Stage Predictions

Now we predict all 72 group stage fixtures.

In [ ]:
from predict_wc2026 import predict_group_stage, load_actual_results, merge_actual_results

match_data = predict_group_stage(xgb, temp_cal, bp, ensemble, all_data, quiet=True)
actual = load_actual_results()
match_data = merge_actual_results(match_data, actual)

# Show Group I
print('Group I — France, Senegal, Iraq, Norway')
print(f'{"Match":<30} {"Home win":>10} {"Draw":>8} {"Away win":>10}')
print('-' * 62)
for m in [m for m in match_data if m.get('group') == 'I']:
    label = f"{m['home_team']} vs {m['away_team']}"
    print(f'{label:<30} {m["p_home"]:>9.1%} {m["p_draw"]:>8.1%} {m["p_away"]:>9.1%}')

---
## Step 8 — Tournament Win Probabilities

We simulate the entire tournament 50,000 times and count how often each team wins.
This takes ~90 seconds.

In [ ]:
import subprocess, os

# 50,000 simulations — same as the default. Takes ~90 seconds.
# Drop to --sims 10000 if you just want a quick preview.
result = subprocess.run(
    [sys.executable, os.path.join(project_root, 'scripts', 'predict_wc2026.py'),
     '--sims', '50000', '--quiet'],
    capture_output=True, text=True, cwd=project_root
)

in_table = False
for line in result.stdout.split('\n'):
    if 'TOURNAMENT PROGRESSION' in line:
        in_table = True
    if in_table:
        print(line)

---
## Step 9 — Recording Actual Results

As the World Cup plays out, feed in real results from your terminal:

```bash
# Record a result
python3.11 scripts/update_wc2026.py --result "France vs Norway" --score "2-1" --group I

# List all recorded results
python3.11 scripts/update_wc2026.py list
```

The next prediction run will:
- Lock that match to the actual score
- Update the Kalman filter with the new information
- Re-simulate only the remaining unplayed fixtures

---
## Step 10 — Competition Submission

Generate `output/output.csv` with one predicted score per match, optimised for competition points:

In [ ]:
import pandas as pd

submission = pd.read_csv(os.path.join(project_root, 'output', 'output.csv'))
print(f'Submission: {len(submission)} matches')
submission

To regenerate (e.g. after recording new results):
```bash
python3.11 scripts/generate_submission.py
```

---
## Step 11 — How Good Is the Model?

We backtest by training on pre-WC-2022 data and testing on the 64 WC 2022 matches.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

metrics = Path(project_root) / 'output' / 'backtest_2022_metrics.txt'
if metrics.exists():
    print(metrics.read_text())
else:
    print('No backtest results yet. Run from terminal:')
    print('  python3.11 scripts/backtest.py --years 2022')

In [ ]:
cal_plot = Path(project_root) / 'output' / 'backtest_2022_calibration.png'
if cal_plot.exists():
    display(Image(filename=str(cal_plot)))
else:
    print('Run backtest first to generate the calibration plot.')

---
## Summary

| What | How |
|---|---|
| **Data** | 47,000+ international matches, match-importance weighted |
| **Training features** | ~120 per match from 13 modules (after correlation pruning) |
| **WC 2026 context** | 5 post-processing modules: venue, sofifa, api_form, squad list, injury |
| **Model** | XGBoost + Temperature Scaling + BayesPoisson (DC ρ) + Context-Adaptive Ensemble |
| **Score prediction** | Expected-value optimisation over competition scoring rules |
| **Third-place advancement** | Cross-group P(advance) simulation in generate_submission_v2.py |
| **Live updates** | Record actual results → Kalman EKF update → re-simulate remaining matches |
| **Tournament simulator** | 50,000 Monte Carlo sims with Kalman posterior λ resampling |

**Key scripts:**

| Script | What it does |
|---|---|
| `pipeline.py` | Full end-to-end pipeline (preferred entry point) |
| `predict_wc2026.py` | Direct prediction + tournament table |
| `generate_submission_v2.py` | Competition output.csv (third-place aware) |
| `update_wc2026.py` | Record actual WC 2026 results |
| `backtest.py --years 2014 2018 2022 --continental` | Model quality evaluation with bootstrap CI |
| `calibrate_confederations.py` | Data-driven confederation strength derivation |